In [15]:
import geopandas as gpd
import polars as pl
import polars_h3 as plh3
import yaml
from datetime import datetime

with open("../config.yaml") as fp:
    config = yaml.safe_load(fp)

In [2]:
PARTITION = 0
DIR = f"{config['dirs']['output']}/stops/"
OUTPUT = "../output/"

In [3]:
data = pl.read_parquet(f"{DIR}/{PARTITION:02x}.parquet")
data.head(3)

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label
i64,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64
1,1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,17
1,2,2024-10-05 11:06:43 CEST,2024-10-05 11:22:57 CEST,16m 14s,47.498402,19.0404,0.0,0.0,4,14
1,3,2024-10-05 17:49:22 CEST,2024-10-05 17:59:54 CEST,10m 32s,47.476742,19.075131,0.0,0.0,3,8


In [4]:
print(data.head(3).to_pandas().to_markdown(index=False))

|   device_id |   stop_id | interval_start            | interval_stop             | interval_lenght   |   latitude_mean |   longitude_mean |   latitude_std |   longitude_std |   number_of_pings |   place_label |
|------------:|----------:|:--------------------------|:--------------------------|:------------------|----------------:|-----------------:|---------------:|----------------:|------------------:|--------------:|
|           1 |         1 | 2024-10-02 07:24:06+02:00 | 2024-10-02 07:35:29+02:00 | 0 days 00:11:23   |         47.6612 |          19.0764 |              4 |               0 |                 2 |            17 |
|           1 |         2 | 2024-10-05 11:06:43+02:00 | 2024-10-05 11:22:57+02:00 | 0 days 00:16:14   |         47.4984 |          19.0404 |              0 |               0 |                 4 |            14 |
|           1 |         3 | 2024-10-05 17:49:22+02:00 | 2024-10-05 17:59:54+02:00 | 0 days 00:10:32   |         47.4767 |          19.0751 |            

In [7]:
data = data.with_columns(
    plh3.int_to_str(
        plh3.latlng_to_cell(pl.col("latitude_mean"), pl.col("longitude_mean"), 8)
    ).alias("h3_8"),
    plh3.int_to_str(
        plh3.latlng_to_cell(pl.col("latitude_mean"), pl.col("longitude_mean"), 9)
    ).alias("h3_9"),
    plh3.int_to_str(
        plh3.latlng_to_cell(pl.col("latitude_mean"), pl.col("longitude_mean"), 10)
    ).alias("h3_10"),
    pl.col("interval_start").dt.date().alias("day"),
    pl.col("interval_start").dt.time().alias("time"),
    pl.col("interval_start").dt.weekday().alias("dow"),
)

In [8]:
data.head(3)

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label,h3_8,h3_9,h3_10,day,time,dow
i64,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64,str,str,str,date,time,i8
1,1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,17,"""881e037261fffff""","""891e037260bffff""","""8a1e0372609ffff""",2024-10-02,07:24:06,3
1,2,2024-10-05 11:06:43 CEST,2024-10-05 11:22:57 CEST,16m 14s,47.498402,19.0404,0.0,0.0,4,14,"""881e037853fffff""","""891e0378537ffff""","""8a1e03785377fff""",2024-10-05,11:06:43,6
1,3,2024-10-05 17:49:22 CEST,2024-10-05 17:59:54 CEST,10m 32s,47.476742,19.075131,0.0,0.0,3,8,"""881e0378c1fffff""","""891e0378c0bffff""","""8a1e0378c087fff""",2024-10-05,17:49:22,6


In [9]:
h3_8_count = data.group_by(["h3_8"]).len().rename({"h3_8": "h3", "len": "count"})
h3_9_count = data.group_by(["h3_9"]).len().rename({"h3_9": "h3", "len": "count"})
h3_10_count = data.group_by(["h3_10"]).len().rename({"h3_10": "h3", "len": "count"})

In [18]:
with open("06_home_detection_config.yaml") as fp:
    hd_config = yaml.safe_load(fp)

In [20]:
def detect_home_work_pl(
    data: pl.DataFrame,
    config: dict,
    spatial_id: str = "h3",
    device_id: str = "device_id",
) -> pl.DataFrame:
    home_hours_end = datetime.strptime(config["home_hours"]["end"], "%H:%M").time()
    home_hours_begin = datetime.strptime(config["home_hours"]["begin"], "%H:%M").time()
    work_hours_end = datetime.strptime(config["work_hours"]["end"], "%H:%M").time()
    work_hours_begin = datetime.strptime(config["work_hours"]["begin"], "%H:%M").time()

    home_stay_count = (
        data.filter(
            (
                (
                    (pl.col("time") < home_hours_end)
                    | (pl.col("time") > home_hours_begin)
                )
                & (pl.col("dow") < 6)
            )
            | ((pl.col("dow") >= 6) | (pl.col("day").is_in(config["weekday_holidays"])))
        )
        .group_by([device_id, spatial_id])
        .agg(pl.col("day").count())
        .rename({"day": "count"})
        .sort([device_id, "count"], descending=[False, True])
        .unique(subset=device_id, keep="first")
    )
    work_stay_count = (
        data.filter(
            ((pl.col("time") > work_hours_begin) & (pl.col("time") < work_hours_end))
            & ((pl.col("dow") < 6) | (pl.col("day").is_in(config["weekend_workdays"])))
            & (~pl.col("day").is_in(config["weekday_holidays"]))
        )
        .group_by([device_id, spatial_id])
        .agg(pl.col("day").count())
        .rename({"day": "count"})
        .sort([device_id, "count"], descending=[False, True])
        .unique(subset=device_id, keep="first")
    )
    return home_stay_count.join(
        work_stay_count,
        on=device_id,
        how="full",
        suffix="_work",
        coalesce=True,
    ).rename({spatial_id: f"{spatial_id}_home", "count": "count_home"})

In [21]:
hw8 = detect_home_work_pl(data, hd_config, spatial_id="h3_8", device_id="device_id")
hw8.write_csv(f"{OUTPUT}/home_work_{PARTITION:02x}_h3_8.csv")
hw9 = detect_home_work_pl(data, hd_config, spatial_id="h3_9", device_id="device_id")
hw9.write_csv(f"{OUTPUT}/home_work_{PARTITION:02x}_h3_9.csv")
hw10 = detect_home_work_pl(data, hd_config, spatial_id="h3_10", device_id="device_id")
hw10.write_csv(f"{OUTPUT}/home_work_{PARTITION:02x}_h3_10.csv")

Detect home in settlements

In [22]:
settlements = gpd.read_file("../data/osm/hungary/settlements.geojson").set_crs(4326)
settlements = settlements[settlements["name"] != "Budapest"].copy()

In [34]:
def map_to_settlements(
    data: pl.DataFrame, settlements: gpd.GeoDataFrame
) -> pl.DataFrame:
    gdf = data.to_pandas()  # 9s
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf["longitude_mean"], gdf["latitude_mean"]),
        crs=4326,
    )
    # +23s
    gdf = gdf.sjoin(settlements).drop(["index_right", "geometry"], axis=1)
    # + 90s
    gdf = pl.from_pandas(gdf).with_columns(pl.col("day").dt.date())
    # +13s

    return gdf

In [35]:
data_s = map_to_settlements(data, settlements)
# runs 2m 18s for p00

In [36]:
data_s.head(3)

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label,h3_8,h3_9,h3_10,day,time,dow,osm_id,name
i64,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64,str,str,str,date,time,i8,i32,str
1,1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,17,"""881e037261fffff""","""891e037260bffff""","""8a1e0372609ffff""",2024-10-02,07:24:06,3,216160,"""Szentendre"""
1,2,2024-10-05 11:06:43 CEST,2024-10-05 11:22:57 CEST,16m 14s,47.498402,19.0404,0.0,0.0,4,14,"""881e037853fffff""","""891e0378537ffff""","""8a1e03785377fff""",2024-10-05,11:06:43,6,221984,"""I. kerület"""
1,3,2024-10-05 17:49:22 CEST,2024-10-05 17:59:54 CEST,10m 32s,47.476742,19.075131,0.0,0.0,3,8,"""881e0378c1fffff""","""891e0378c0bffff""","""8a1e0378c087fff""",2024-10-05,17:49:22,6,1552462,"""IX. kerület"""


In [37]:
hws = detect_home_work_pl(data_s, hd_config, spatial_id="name", device_id="device_id")
hws.write_csv(f"{OUTPUT}/home_work_{PARTITION:02x}_settlement.csv")